# Dispatch Model Fitting & Analysis for a Table of Hyperparameters to SLURM


In [149]:
from types import SimpleNamespace
import os
from datetime import datetime
import logging
from shortuuid import uuid
from nbconvert import PythonExporter
import nbformat
import pandas as pd
from tqdm import tqdm
import slurm_requests as slurm

from arg_io import ArgType, ArgDict, argd_to_argv
from parameter_table_generator import (
    make_grid_scan_table,
    unique_values_per_column,
    iter_rows,
)

## enable logging


In [150]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## SLURM (for Maxwell @ DESY)


### user


In [151]:
def get_user_name():
    """
    Obtain user name.

    **WARNING**: This version is specific for Maxwell @ DESY and requires this notebook to be run from within the cluster. Adapt this function if needed.
    """
    o = !whoami
    user_name = o[0]
    return user_name

In [152]:
def get_current_working_directory():
    """
    Obtain current working directory.
    """
    o = !pwd
    working_directory = o[0]
    return working_directory

### token management


In [153]:
def get_new_slurm_user_token():
    """
    Obtain a new SLURM user token valid for 24 hours.

    **WARNING**: This version is specific for Maxwell @ DESY and requires this notebook to be run from within the cluster. Adapt this function if needed.
    """
    o = !/software/tools/bin/slurm_token -l $((3600*24))
    token = o[0].split("=")[1]
    return token


### init


In [154]:
slurm.init_defaults(
    url="https://max-slurm-rest.desy.de/sapi",
    api_version="v0.0.40",
    user_name=get_user_name(),
    user_token=get_new_slurm_user_token(),
    partition="maxgpu",
    environment=[
        "PATH=/bin:/usr/bin/:/usr/local/bin/",
        "LD_LIBRARY_PATH=/lib/:/lib64/:/usr/local/lib",
    ],
)

### refresh token


In [155]:
def refresh_slurm_user_token():
    slurm.default.user_token = get_new_slurm_user_token()

### test connection


In [156]:
_ = await slurm.ping()
_ = await slurm.diagnose()

## aux


### UUID


In [157]:
def get_uuid() -> str:
    """Generate a new UUID based on the current date and time."""
    return f"{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}-{uuid()[:8]}"

### scripts


In [158]:
def convert_notebook(nb_path: str, py_path: str) -> None:
    with open(nb_path, "r", encoding="utf-8") as f:
        notebook_node = nbformat.read(f, as_version=4)

    python_exporter = PythonExporter()
    python_code, _ = python_exporter.from_notebook_node(notebook_node)

    with open(py_path, "w", encoding="utf-8") as f:
        f.write(python_code)

    logger.info(f"Converted '{nb_path}' to '{py_path}'.")


def convert_notebooks(
    working_directory: str,
    fit_py: str,
    eval_py: str,
) -> None:
    os.makedirs(os.path.dirname(fit_py), exist_ok=True)
    os.makedirs(os.path.dirname(eval_py), exist_ok=True)
    convert_notebook(
        nb_path=os.path.join(working_directory, "2_model_fitting.ipynb"),
        py_path=fit_py,
    )
    convert_notebook(
        nb_path=os.path.join(working_directory, "3_model_evaluation.ipynb"),
        py_path=eval_py,
    )

In [159]:
def escape(s: ArgType) -> str:
    """
    convert to str
    quote strings with spaces if needed
    """
    if isinstance(s, str):
        return f'"{s}"' if " " in s else s
    return str(s)

In [160]:
def bash_script(
    *,
    working_directory: str,
    cmd: str,
) -> str:
    """
    Wraps a command in a bash script with environment setup.

    This setup targets the Maxwell cluster at DESY as of October 2025.
    """

    return f"""#!/usr/bin/env bash

# safer and verbose execution (e.g. immediate exit of fail)
# -e exit immediately when a command fails
# -u treat unset variables as an error
# -o pipelfail set exit code based on entire pipe
set -euo pipefail

# setup environment
unset LD_PRELOAD
source /etc/profile.d/modules.sh
module purge
module load maxwell python

cd "{working_directory}"

source venv/bin/activate

PYTHONPATH=.
export PYTHONPATH

{cmd}
"""

### submission


In [161]:
def pre_submission_routine(working_directory: str):
    # refresh SLURM token
    refresh_slurm_user_token()

    # convert notebooks to scripts
    script_uid = get_uuid()
    tmp_path = os.path.join(working_directory, "data", "tmp", "scripts")
    fit_py = os.path.join(tmp_path, f"model_fitting-{script_uid}.py")
    eval_py = os.path.join(tmp_path, f"model_evaluation-{script_uid}.py")

    convert_notebooks(working_directory, fit_py, eval_py)

    return dict(fit=fit_py, eval=eval_py)

## Parameter Table Aux


In [162]:
def print_unique_values_per_column(df: pd.DataFrame) -> None:
    """Print unique values per column in a DataFrame."""
    uv = unique_values_per_column(df)
    for col, values in uv.items():
        values = sorted(values)
        print(
            f"{col}  ({values[0]:0.3g}...{values[-1]:0.3g}, count={len(values)}): "
            + " ".join(map(lambda x: f"{x:0.3g}", values))
        )

## Main


### Args

#### Demo

In [163]:
# args = SimpleNamespace(
#     # io
#     working_directory=get_current_working_directory(),
#     data_path="data/demo",
#     logs_path="data/logs",
#     experiment_name="demo_experiment",
#     slurm_path=os.path.join(get_current_working_directory(), "data", "tmp", "slurm"),
#     # fit
#     fit_num_workers=4,
#     fit_persistent_workers=True,
#     fit_pin_memory=True,
#     fit_timeout=1 * 60 * 60,  # seconds
#     # eval
#     eval_batch_size=128,
#     eval_timeout=30 * 60,  # seconds
# )

In [ ]:
args = SimpleNamespace(
    # io
    working_directory=get_current_working_directory(),
    data_path="data/laser-pulse-shaping-astra-sim-v11-normalized",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v11-normalized",
    slurm_path=os.path.join(get_current_working_directory(), "data", "tmp", "slurm"),
    # fit
    fit_num_workers=4,
    fit_persistent_workers=True,
    fit_pin_memory=True,
    fit_timeout=4 * 60 * 60,  # seconds
    # eval
    eval_batch_size=128,
    eval_timeout=30 * 60,  # seconds
)

### cmd argds


In [ ]:
def fit_argd(
    *,
    version: str,
    hidden_layer_sizes: list[int],
    learning_rate: float,
    batch_size: int,
) -> ArgDict:
    return dict(
        # io
        data_path=args.data_path,
        logs_path=args.logs_path,
        experiment_name=args.experiment_name,
        version=version,
        # data
        batch_size=batch_size,
        num_workers=args.fit_num_workers,
        persistent_workers=args.fit_persistent_workers,
        pin_memory=args.fit_pin_memory,
        # model
        hidden_layer_sizes=hidden_layer_sizes,
        leaky_relu_factor=0.1,
        # optimizer
        learning_rate=learning_rate,
        weight_decay=0.0,
        # trainer
        max_steps=1_000_000,
        log_every_n_steps=10,
        accelerator="auto",
        # devices=None,
        # profiler=None,
        # callbacks
        early_stopping_val_train_ratio_training_window_size=1000,
        early_stopping_val_train_ratio_validation_window_size=100,
        early_stopping_val_train_ratio_upper_threshold=1.2,
        early_stopping_val_train_ratio_bad_epochs_limit=100,
    )

In [166]:
def eval_argd(
    *,
    version: str,
) -> ArgDict:
    return dict(
        # io
        data_path=args.data_path,
        logs_path=args.logs_path,
        experiment_name=args.experiment_name,
        version=version,
        # data
        batch_size=args.eval_batch_size,
    )

### job submission


In [167]:
async def submit_run(
    fit_py: str,
    eval_py: str,
    *,
    width: int,
    depth: int,
    learning_rate: float,
    batch_size: int,
) -> tuple[int, int]:

    version = get_uuid()
    hidden_layer_sizes = [width] * depth

    # fitting
    argd = fit_argd(
        version=version,
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate=learning_rate,
        batch_size=batch_size,
    )
    fitting_job_id = await submit_part(
        version=version,
        part="fit",
        part_py=fit_py,
        argd=argd,
        timeout=args.fit_timeout,
    )

    # evaluation
    argd = eval_argd(
        version=version,
    )
    evaluation_job_id = await submit_part(
        version=version,
        part="eval",
        part_py=eval_py,
        argd=argd,
        timeout=args.eval_timeout,
        dependency=f"afterany:{fitting_job_id}",
    )

    return fitting_job_id, evaluation_job_id


async def submit_part(
    *,
    version: str,
    part: str,
    part_py: str,
    argd: ArgDict,
    timeout: int,  # seconds
    dependency: str | None = None,
) -> int:

    name = f"{part}-{args.experiment_name}-{version}"
    cmd_args = " ".join(map(escape, argd_to_argv(argd)))
    cmd = f'python "{part_py}" {cmd_args}'
    job_id, _ = await slurm.job_submit(
        name=name,
        working_directory=args.working_directory,
        script=bash_script(
            working_directory=args.working_directory,
            cmd=cmd,
        ),
        stdout=os.path.join(args.slurm_path, f"{name}-%j.out"),
        stderr=os.path.join(args.slurm_path, f"{name}-%j.err"),
        time_limit=timeout // 60,  # minutes
        dependency=dependency,
    )

    if job_id is None:
        raise RuntimeError(f"Failed to submit job '{name}'.")

    return job_id

### parameter table


Logbook:

- 2025-12-10:
  - large scale scan for `laser-pulse-shaping-astra-sim-v11-normalized`


In [ ]:
# large scale
scan_table = make_grid_scan_table(
    [
        {"width": [10, 50, 100, 200]},
        {"depth": range(1, 6)},
        {"learning_rate": [1e-1, 1e-2, 1e-3, 1e-4]},
        {"batch_size": [16, 32, 64, 128]},
    ]
)

# for testing
# scan_table = make_grid_scan_table(
#     [
#         {"width": [10]},
#         {"depth": range(1, 2)},
#         {"learning_rate": [1e-2]},
#         {"batch_size": [64]},
#     ]
# )


scan_table.drop_duplicates(inplace=True)

# show summary
print(f"{len(scan_table)} simulations to run.")
print(
    f"cumulative simulation timeout: {len(scan_table)*args.fit_timeout/60/60/24:.1f} days"
)
print()
print_unique_values_per_column(scan_table)

320 simulations to run.
cumulative simulation timeout: 13.3 days

width  (10...200, count=4): 10 50 100 200
depth  (1...5, count=5): 1 2 3 4 5
learning_rate  (0.0001...0.1, count=4): 0.0001 0.001 0.01 0.1
batch_size  (16...128, count=4): 16 32 64 128


### dispatch


In [169]:
scripts = pre_submission_routine(args.working_directory)

job_ids = []
for params in tqdm(iter_rows(scan_table), total=len(scan_table)):
    ids = await submit_run(
        fit_py=scripts["fit"],
        eval_py=scripts["eval"],
        **params,
    )
    job_ids += list(ids)

INFO:__main__:Converted '/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/2_model_fitting.ipynb' to '/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/data/tmp/scripts/model_fitting-2025-12-10-16-43-49-YdkFadNF.py'.
INFO:__main__:Converted '/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/3_model_evaluation.ipynb' to '/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/data/tmp/scripts/model_evaluation-2025-12-10-16-43-49-YdkFadNF.py'.
100%|██████████| 320/320 [00:07<00:00, 45.32it/s]
